# Telco Customer Churn — Exploratory Data Analysis

This notebook explores the Telco Customer Churn dataset to understand feature distributions, class imbalance, and relationships with the target variable.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load raw data
df = pd.read_csv(Path("..") / "data" / "raw" / "Telco-Customer-Churn.csv")
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
# Dataset overview
print("=" * 60)
print("SHAPE:", df.shape)
print("=" * 60)
print("\nINFO:")
df.info()
print("\n" + "=" * 60)
print("HEAD:")
display(df.head())
print("\n" + "=" * 60)
print("DESCRIBE:")
display(df.describe(include="all"))

In [ ]:
# Missing values analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap of missing values
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, ax=axes[0], cmap="viridis")
axes[0].set_title("Missing Values Heatmap")

# Count of missing values per column
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    axes[1].text(0.5, 0.5, "No null values found", ha="center", va="center", fontsize=14)
else:
    missing.plot(kind="bar", ax=axes[1])
axes[1].set_title("Missing Values Count")

plt.tight_layout()
plt.show()

# TotalCharges issue — blank strings that should be NaN
print("\nTotalCharges dtype:", df["TotalCharges"].dtype)
blank_total_charges = df[df["TotalCharges"] == " "]
print(f"Blank TotalCharges rows: {len(blank_total_charges)}")
if len(blank_total_charges) > 0:
    print("These rows have tenure=0 (new customers):")
    display(blank_total_charges[["customerID", "tenure", "TotalCharges", "Churn"]])

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["Churn"].value_counts()
pcts = df["Churn"].value_counts(normalize=True) * 100
bars = ax.bar(counts.index, counts.values, color=["steelblue", "coral"])
for bar, pct in zip(bars, pcts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f"{pct:.1f}%", ha="center", fontweight="bold")
ax.set_title("Churn Class Distribution")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()
print(f"\nImbalance ratio: {counts['No'] / counts['Yes']:.2f}:1")

In [ ]:
# Numeric feature distributions by churn status
# Fix TotalCharges for plotting
df_plot = df.copy()
df_plot["TotalCharges"] = pd.to_numeric(df_plot["TotalCharges"], errors="coerce")

numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, numeric_cols):
    for label, color in [("No", "steelblue"), ("Yes", "coral")]:
        subset = df_plot[df_plot["Churn"] == label][col].dropna()
        ax.hist(subset, bins=30, alpha=0.5, label=f"Churn={label}", color=color, density=True)
        subset.plot.kde(ax=ax, color=color, linewidth=2)
    ax.set_title(f"{col} by Churn")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Categorical feature churn rates
cat_cols = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
]

n_cols = 4
n_rows = (len(cat_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)["Churn"].apply(lambda x: (x == "Yes").mean())
    churn_rate.plot(kind="bar", ax=axes[i], color="coral")
    axes[i].set_title(f"{col} Churn Rate")
    axes[i].set_ylabel("Churn Rate")
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis="x", rotation=45)

# Hide unused axes
for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis
df_corr = df_plot.copy()
df_corr["Churn_binary"] = (df_corr["Churn"] == "Yes").astype(int)

numeric_for_corr = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "Churn_binary"]
corr_matrix = df_corr[numeric_for_corr].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlation Matrix (Numeric Features + Target)")
plt.tight_layout()
plt.show()

# Point-biserial correlation with target
from scipy.stats import pointbiserialr

print("\nPoint-biserial correlation with Churn:")
print("-" * 40)
for col in ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]:
    valid = df_corr[[col, "Churn_binary"]].dropna()
    corr, pval = pointbiserialr(valid["Churn_binary"], valid[col])
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"  {col:20s}  r={corr:+.3f}  p={pval:.2e} {sig}")

## Key Findings

1. **Class Imbalance**: ~26.5% churn rate — the dataset is imbalanced (roughly 2.7:1 ratio). Models should use `scale_pos_weight` or `class_weight="balanced"`.

2. **TotalCharges Issue**: 11 rows have blank `TotalCharges` (all with `tenure=0`). These are new customers with no billing history — handled during data cleaning via median imputation.

3. **Strong Churn Predictors**:
   - **Contract type**: Month-to-month customers churn at ~43% vs ~3% for two-year contracts
   - **Internet service**: Fiber optic users churn more (~42%) than DSL (~19%) or no-internet (~7%)
   - **tenure**: Churners have significantly shorter tenure (negative correlation)
   - **MonthlyCharges**: Higher charges correlate with higher churn (positive correlation)
   - **Add-on services** (OnlineSecurity, TechSupport, etc.): Customers without these services churn more

4. **Weak Predictors**: `gender` and `PhoneService` show negligible difference in churn rates.

5. **Feature Engineering Opportunities**: Contract type and internet service type are dominant — consider interaction features or ordinal encoding for Contract.